# 12 - Integrated PUNDIT case study: TK-101

**Week 13; suggested duration: two teaching periods**

## Learning objectives

- Integrate the methods into a traceable workflow
- Execute a qualitative/formal/dynamic/decision chain
- Separate results from limitations
- Specify a manageable student project

**PUNDIT connection.** This unit supports the educational exploitation of PUNDIT (`2020-1.2.3-EUREKA-2022-00021`). The teaching material alone does not establish fulfillment of the original contractual technical deliverables. All TK-101 numerical inputs are synthetic unless explicitly stated otherwise.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Repository root:", ROOT)


## 1. Integration map

**system -> hazards -> failure modes -> PHA/FMEA/HAZOP -> FTA -> Bayesian evidence / Markov dynamics -> Monte Carlo uncertainty -> diagnostic update -> RBM decision**.

The objective is not to use every method simultaneously, but to understand which question each answers and how information can pass between them.


## 2. Native SysML system model and hazards

The new path starts from the reference-tool JSON export of `models/sysml/tk101.sysml`. The adapter resolves model elements and Boolean hazard expressions into the existing exchange schema. The legacy YAML is retained as a comparison fixture, not relabeled as native SysML.


In [ ]:
from safetycourse.sysml_adapter import load_sysml_export
model=load_sysml_export(ROOT/'models/sysml/tk101.export.json')
print(model['metadata']); print('components:',[c['id'] for c in model['components']]); print('hazards:',[h['id'] for h in model['hazards']])

## 3. Qualitative chain

PHA: overflow/fire. FMEA: LT failed low, LSHH/XV fail on demand, operator miss. HAZOP: MORE LEVEL and NO EFFECTIVE SHUTDOWN. Engineering analysis supplies the hazard rules encoded in the SysML model.


In [ ]:
qual=pd.DataFrame([
 {'PHA':'Overflow','FMEA':'LT-101 failed low','HAZOP':'MORE LEVEL','formal':'FTA basic event'},
 {'PHA':'Overflow','FMEA':'LSHH fail on demand','HAZOP':'NO SHUTDOWN','formal':'FTA basic event'},
 {'PHA':'Overflow','FMEA':'XV fail on demand','HAZOP':'NO SHUTDOWN','formal':'FTA basic event'}]); qual


## 4. Model -> FTA -> baseline probability

Use the package to generate a fault tree, enumerate minimal cut sets and calculate the top-event probability under the independent-basic-event model.


In [ ]:
from safetycourse.model_io import validate_system_model,generate_fault_tree
from safetycourse.fta import top_event_probability,minimal_cut_sets
m=model; print(validate_system_model(m)); tree=generate_fault_tree(m,'OVERFLOW')
p0=top_event_probability(tree); print('baseline P(OVERFLOW)=',p0); print([sorted(x) for x in minimal_cut_sets(tree)])

## 5. Sensor data -> evidence

Form a diagnostic score from the synthetic LT-101 residual. The score-to-probability mapping is an illustrative assumption requiring calibration for real use.


In [ ]:
df=pd.read_csv(ROOT/'data'/'pundit_case'/'sensor_data.csv'); r=df.lt101_measured_pct-df.true_level_pct
def sigmoid(x): return 1/(1+np.exp(-x))
p_sensor=.002+(.6-.002)*sigmoid(-((r+2.5)/.8))
plt.plot(df.time_min,p_sensor); plt.xlabel('time'); plt.ylabel('P(sensor fault | evidence)'); plt.grid(alpha=.3); plt.show()


## 6. Evidence -> dynamic overflow risk

Substitute a time-dependent diagnostic proxy for the static LT basic-event prior. This illustrates evidence-sensitive risk calculations, not a validated plant-risk forecast.


In [ ]:
p_ch=.08; p_auto=1-(1-.01)*(1-.005); p_op=.04
p_dynamic=p_ch*p_auto*(1-(1-p_sensor)*(1-p_op))
plt.semilogy(df.time_min,p_dynamic); plt.axhline(p0,ls='--',label='baseline model'); plt.legend(); plt.xlabel('time'); plt.ylabel('P(overflow)'); plt.grid(alpha=.3); plt.show()


## 7. Markov perspective

The state model describes degradation and recovery. It does not answer the same question as static FTA. Its probabilities can only be combined with FTA outputs after defining a consistent event and time-horizon mapping.


In [ ]:
from scipy.linalg import expm
states=['S0','S1','S2','S3','S4','S5']; Q=np.zeros((6,6))
def add(i,j,x): Q[i,j]+=x
add(0,1,.002); add(0,2,.001); add(0,3,.004); add(1,0,.08); add(1,3,.004); add(2,0,.05); add(2,3,.004); add(3,0,.2); add(3,4,.01); add(4,0,.1); add(4,5,.03)
for i in range(6): Q[i,i]=-Q[i].sum()
p=np.array([1,0,0,0,0,0.])@expm(Q*100); print(dict(zip(states,p))); print('P(S4 or S5)=',p[4]+p[5])


## 8. Monte Carlo uncertainty

Illustrate uncertainty in baseline basic-event probabilities using synthetic Beta parameters.


In [ ]:
rng=np.random.default_rng(9); N=20_000
pch=rng.beta(8,92,N); pl=rng.beta(2,198,N); px=rng.beta(1,199,N); ps=rng.beta(4,196,N); po=rng.beta(8,192,N)
p_mc=pch*(1-(1-pl)*(1-px))*(1-(1-ps)*(1-po))
print('5/50/95%=',np.quantile(p_mc,[.05,.5,.95]))


## 9. Risk-based maintenance decision

The motivating question is: **maintain LT-101 now, or wait for the next opportunity window?** These synthetic cost/probability inputs illustrate a decision structure, not an approved operational recommendation.


In [ ]:
Cc=50_000_000; Cm=250_000; prod_now=800_000; prod_window=100_000; p_after=.001
cost_now=Cm+prod_now+p_after*Cc; current=float(p_dynamic.iloc[-1] if hasattr(p_dynamic,'iloc') else p_dynamic[-1]); cost_wait=Cm+prod_window+current*Cc
pd.DataFrame([{'option':'maintain now','expected_cost':cost_now},{'option':'wait','expected_cost':cost_wait}])


## 10. Traceability and PUNDIT educational exploitation

Record source artifacts, model versions, synthetic/real data status, assumptions, generated artifacts, validation evidence, chapter and project ID. `technical_deliverable_claim: false` means this teaching repository does not itself certify contractual acceptance. The adapter's demonstrated technical capability is documented separately in `docs/sysml_adapter.md`.


In [ ]:
trace=pd.read_csv(ROOT/'data'/'pundit_case'/'sensor_data.csv').head(1)
print('PUNDIT project: 2020-1.2.3-EUREKA-2022-00021')
print('use: educational_exploitation')
print('technical_deliverable_claim: false')


## 11. Semester project

Select a small safety-critical subsystem. Include system boundaries; PHA/FMEA/HAZOP; a formal risk model; a parameterized Python notebook; assumptions and uncertainty; a maintenance/mitigation decision; and traceability. The quality of the engineering reasoning matters more than model size.


## Try it!

1. Choose a TK-101 basic event and design a data source for probability calibration.
2. Compare static FTA probability with Markov S4+S5 occupancy probability. Why are these not identical?
3. Add a common-cause event and document its source traceability.
4. Write a five-sentence decision memo for maintain-now versus wait, including assumptions.


## Summary

The course connects **system model -> hazard -> formal model -> evidence -> dynamic risk -> decision** in a traceable engineering chain. PUNDIT educational exploitation and the restricted adapter demonstration are documented; industrial validation and contractual acceptance remain separate questions.
